# LLM Classification Fine-tuning

Predicting user preferences in Chatbot Arena head-to-head LLM battles using a preference/reward model approach.

## 1. Setup and Imports

In [ ]:
# Standard library
import os
os.environ['HF_HUB_OFFLINE'] = '1'  # Fail fast if model not cached locally
import sys
import json
import re
import logging
import warnings
import random
import string
from pathlib import Path
from collections import Counter

# Data processing
import pandas as pd
import numpy as np

# ML frameworks
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
import lightgbm as lgb
import xgboost as xgb
import optuna
from scipy.optimize import minimize

# NLP — sentence_transformers is optional (needs internet to download model)
try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False

# NLTK — try to use pre-installed data, fall back to simple alternatives
import nltk
try:
    nltk.data.find('tokenizers/punkt_tab')
    HAS_NLTK_PUNKT = True
except LookupError:
    try:
        nltk.download('punkt', quiet=True)
        nltk.download('punkt_tab', quiet=True)
        HAS_NLTK_PUNKT = True
    except Exception:
        HAS_NLTK_PUNKT = False

try:
    nltk.data.find('corpora/stopwords')
    HAS_NLTK_STOPWORDS = True
except LookupError:
    try:
        nltk.download('stopwords', quiet=True)
        HAS_NLTK_STOPWORDS = True
    except Exception:
        HAS_NLTK_STOPWORDS = False

# Sentence tokenizer: use nltk if available, else regex fallback
def sent_tokenize(text):
    """Split text into sentences. Uses NLTK if available, else regex."""
    if HAS_NLTK_PUNKT:
        return nltk.sent_tokenize(text)
    # Simple regex fallback: split on .!? followed by space or end
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    return [s for s in sentences if s]

# Stopwords: use nltk if available, else hardcoded common English stopwords
if HAS_NLTK_STOPWORDS:
    STOPWORDS = set(nltk.corpus.stopwords.words('english'))
else:
    STOPWORDS = {
        'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your',
        'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her',
        'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs',
        'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those',
        'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had',
        'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if',
        'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with',
        'about', 'against', 'between', 'through', 'during', 'before', 'after', 'above',
        'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under',
        'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why',
        'how', 'all', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such',
        'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's',
        't', 'can', 'will', 'just', 'don', 'should', 'now', 'd', 'll', 'm', 'o', 're',
        've', 'y', 'ain', 'aren', 'couldn', 'didn', 'doesn', 'hadn', 'hasn', 'haven',
        'isn', 'ma', 'mightn', 'mustn', 'needn', 'shan', 'shouldn', 'wasn', 'weren',
        'won', 'wouldn',
    }

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import shap

# Utilities
import yaml
from tqdm.auto import tqdm
import joblib

# Configuration
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Project root — on Kaggle use /kaggle/working, otherwise parent of notebooks/
IS_KAGGLE = Path('/kaggle/working').exists()
if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path(os.path.abspath('..'))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"NLTK punkt available: {HAS_NLTK_PUNKT}")
print(f"NLTK stopwords available: {HAS_NLTK_STOPWORDS}")
print(f"Sentence Transformers available: {HAS_SENTENCE_TRANSFORMERS}")
print("All imports loaded successfully.")

In [ ]:
# --- Configuration Loading ---

def load_config(config_name: str) -> dict:
    """Load a YAML config file from the configs directory, return None if not found."""
    config_path = PROJECT_ROOT / 'configs' / f'{config_name}.yaml'
    if config_path.exists():
        with open(config_path, 'r') as f:
            return yaml.safe_load(f)
    return None

# Try loading YAML configs; fall back to inline defaults (for Kaggle)
paths_cfg = load_config('paths') or {
    'kaggle_data_dir': '/kaggle/input/competitions/llm-classification-finetuning',
    'data_dir': 'data',
    'output_dir': 'outputs',
    'train_file': 'train.csv',
    'test_file': 'test.csv',
    'sample_submission': 'sample_submission.csv',
    'log_dir': 'logs',
    'model_dir': 'models',
    'prediction_dir': 'predictions',
    'report_dir': 'reports',
}

features_cfg = load_config('features') or {
    'embedding_model': 'all-MiniLM-L6-v2',
    'embedding_dim': 384,
    'pca_components_response_diff': 50,
    'pca_components_prompt': 20,
    'max_text_length_for_embedding': 512,
    'readability_enabled': True,
    'structural_features_enabled': True,
    'similarity_features_enabled': True,
    'quality_signals_enabled': True,
    'bias_mitigation_enabled': True,
    'hedging_words': ['perhaps', 'maybe', 'might', 'could be', 'possibly', 'arguably', 'it seems', 'I think'],
    'confidence_words': ['certainly', 'definitely', 'clearly', 'obviously', 'undoubtedly', 'absolutely', 'without a doubt'],
    'example_indicators': ['for example', 'e.g.', 'such as', 'for instance', 'consider', 'like'],
}

models_cfg = load_config('models') or {
    'cv_folds': 5,
    'seed': 42,
    'optuna_trials': 50,
    'lgbm': {
        'fixed_params': {'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss', 'verbosity': -1, 'n_jobs': -1},
        'search_space': {
            'n_estimators': [200, 2000], 'learning_rate': [0.01, 0.3], 'max_depth': [3, 10],
            'num_leaves': [15, 127], 'min_child_samples': [5, 100], 'subsample': [0.6, 1.0],
            'colsample_bytree': [0.5, 1.0], 'reg_alpha': [1e-8, 10.0], 'reg_lambda': [1e-8, 10.0],
        },
        'default_params': {
            'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 7, 'num_leaves': 63,
            'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 1.0, 'reg_lambda': 1.0,
        },
    },
    'xgb': {
        'fixed_params': {'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss', 'tree_method': 'hist', 'n_jobs': -1},
        'search_space': {
            'n_estimators': [200, 2000], 'learning_rate': [0.01, 0.3], 'max_depth': [3, 10],
            'min_child_weight': [1, 50], 'subsample': [0.6, 1.0], 'colsample_bytree': [0.5, 1.0],
            'reg_alpha': [1e-8, 10.0], 'reg_lambda': [1e-8, 10.0], 'gamma': [0.0, 5.0],
        },
        'default_params': {
            'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 10,
            'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 1.0, 'reg_lambda': 1.0, 'gamma': 0.1,
        },
    },
    'logistic': {'max_iter': 1000, 'C_range': [0.001, 100.0]},
    'ensemble': {'method': 'weighted_average'},
}

experiment_cfg = load_config('experiment') or {
    'name': 'baseline-v1',
    'seed': 42,
    'debug': False,
    'debug_sample_size': 2000,
}

CONFIG = {
    'paths': paths_cfg,
    'features': features_cfg,
    'models': models_cfg,
    'experiment': experiment_cfg,
}

# Global settings
SEED = CONFIG['experiment']['seed']
DEBUG = CONFIG['experiment']['debug']
DEBUG_SAMPLE_SIZE = CONFIG['experiment']['debug_sample_size']

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)

# Logging
log_dir = PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['log_dir']
log_dir.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_dir / 'training.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Ensure output directories exist
for d in [paths_cfg['model_dir'], paths_cfg['prediction_dir'], paths_cfg['report_dir']]:
    (PROJECT_ROOT / paths_cfg['output_dir'] / d).mkdir(parents=True, exist_ok=True)

logger.info(f"Experiment: {experiment_cfg['name']} | Seed: {SEED} | Debug: {DEBUG}")
print(f"Config loaded: {experiment_cfg['name']}")
print(f"  Seed: {SEED}, Debug: {DEBUG}, CV Folds: {models_cfg['cv_folds']}")

## 2. Data Loading

In [ ]:
# --- Load Raw Data ---
# Use Kaggle input path if available, otherwise fall back to local data dir
kaggle_dir = Path(paths_cfg.get('kaggle_data_dir', ''))
local_dir = PROJECT_ROOT / paths_cfg['data_dir']

if kaggle_dir.exists():
    data_dir = kaggle_dir
    print(f"Using Kaggle data dir: {data_dir}")
else:
    data_dir = local_dir
    print(f"Using local data dir: {data_dir}")

train_df = pd.read_csv(data_dir / paths_cfg['train_file'])
test_df = pd.read_csv(data_dir / paths_cfg['test_file'])

# sample_submission may be in Kaggle dir or local
sample_sub_path = data_dir / paths_cfg['sample_submission']
if not sample_sub_path.exists():
    sample_sub_path = local_dir / paths_cfg['sample_submission']
submission_df = pd.read_csv(sample_sub_path)

if DEBUG:
    train_df = train_df.sample(n=DEBUG_SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    logger.info(f"DEBUG mode: subsampled to {DEBUG_SAMPLE_SIZE} rows")

logger.info(f"Train: {train_df.shape}, Test: {test_df.shape}, Submission: {submission_df.shape}")
print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"Submission format: {submission_df.shape}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Test columns:  {list(test_df.columns)}")
print(f"\nMissing values:\n{train_df.isnull().sum()}")

In [ ]:
# --- Parse JSON Fields and Create Derived Columns ---

def parse_json_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Parse JSON string fields and create derived text columns."""
    df = df.copy()
    
    # Parse JSON lists
    for col in ['prompt', 'response_a', 'response_b']:
        df[f'{col}_parsed'] = df[col].apply(json.loads)
    
    # Number of conversation turns
    df['n_turns'] = df['prompt_parsed'].apply(len)
    
    # Joined full-conversation text
    df['prompt_text'] = df['prompt_parsed'].apply(lambda x: '\n'.join(str(t) for t in x))
    df['response_a_text'] = df['response_a_parsed'].apply(lambda x: '\n'.join(str(t) for t in x))
    df['response_b_text'] = df['response_b_parsed'].apply(lambda x: '\n'.join(str(t) for t in x))
    
    # Last turn only (most relevant for quality assessment)
    df['last_prompt'] = df['prompt_parsed'].apply(lambda x: str(x[-1]) if x else '')
    df['last_response_a'] = df['response_a_parsed'].apply(lambda x: str(x[-1]) if x else '')
    df['last_response_b'] = df['response_b_parsed'].apply(lambda x: str(x[-1]) if x else '')
    
    return df

train_df = parse_json_fields(train_df)
test_df = parse_json_fields(test_df)

# Extract targets
TARGET_COLS = ['winner_model_a', 'winner_model_b', 'winner_tie']
y_train = train_df[TARGET_COLS].values  # shape (N, 3) one-hot
y_labels = np.argmax(y_train, axis=1)   # 0=A wins, 1=B wins, 2=Tie
CLASS_NAMES = ['A wins', 'B wins', 'Tie']

print(f"Parsed {len(train_df)} train rows, {len(test_df)} test rows")
print(f"\nConversation turns: mean={train_df['n_turns'].mean():.2f}, "
      f"median={train_df['n_turns'].median():.0f}, max={train_df['n_turns'].max()}")
print(f"\nTarget distribution:")
for i, name in enumerate(CLASS_NAMES):
    count = (y_labels == i).sum()
    print(f"  {name}: {count} ({count/len(y_labels)*100:.1f}%)")

## 3. Exploratory Data Analysis

In [ ]:
# --- 3a. Class Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Counts
counts = [train_df['winner_model_a'].sum(), train_df['winner_model_b'].sum(), train_df['winner_tie'].sum()]
colors = ['#2196F3', '#FF5722', '#4CAF50']
axes[0].bar(CLASS_NAMES, counts, color=colors)
axes[0].set_title('Outcome Distribution (Counts)')
axes[0].set_ylabel('Count')
for i, (c, name) in enumerate(zip(counts, CLASS_NAMES)):
    axes[0].text(i, c + 200, str(c), ha='center', fontweight='bold')

# Percentages
pcts = [c / len(train_df) * 100 for c in counts]
axes[1].bar(CLASS_NAMES, pcts, color=colors)
axes[1].set_title('Outcome Distribution (%)')
axes[1].set_ylabel('Percentage')
for i, p in enumerate(pcts):
    axes[1].text(i, p + 0.3, f'{p:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['report_dir'] / 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Classes are fairly balanced: ~35/34/31 split.")

In [ ]:
# --- 3b. Model Frequency & Win Rate Analysis (train only) ---
if 'model_a' in train_df.columns:
    all_models = pd.concat([train_df['model_a'], train_df['model_b']]).value_counts()
    top_models = all_models.head(20)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    top_models.plot(kind='barh', ax=ax, color='#5C6BC0')
    ax.set_title('Top 20 Most Frequent Models')
    ax.set_xlabel('Appearances')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Win rate by model (when appearing as model_a)
    model_a_wins = train_df.groupby('model_a')['winner_model_a'].mean().sort_values(ascending=False)
    model_b_wins = train_df.groupby('model_b')['winner_model_b'].mean().sort_values(ascending=False)
    
    # Overall win rate (combine A and B positions)
    win_as_a = train_df.groupby('model_a')['winner_model_a'].agg(['sum', 'count'])
    win_as_b = train_df.groupby('model_b')['winner_model_b'].agg(['sum', 'count'])
    win_as_a.columns = ['wins_a', 'games_a']
    win_as_b.columns = ['wins_b', 'games_b']
    combined = win_as_a.join(win_as_b, how='outer').fillna(0)
    combined['total_wins'] = combined['wins_a'] + combined['wins_b']
    combined['total_games'] = combined['games_a'] + combined['games_b']
    combined['win_rate'] = combined['total_wins'] / combined['total_games']
    combined = combined.sort_values('win_rate', ascending=False)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    top_20_wr = combined.head(20)
    bars = ax.barh(range(len(top_20_wr)), top_20_wr['win_rate'], color='#26A69A')
    ax.set_yticks(range(len(top_20_wr)))
    ax.set_yticklabels(top_20_wr.index)
    ax.set_xlabel('Win Rate')
    ax.set_title('Top 20 Models by Win Rate')
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50% baseline')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print(f"Unique models: {train_df['model_a'].nunique()}")
    print(f"\nTop 5 by win rate:\n{combined['win_rate'].head()}")
    print(f"\nBottom 5 by win rate:\n{combined['win_rate'].tail()}")
else:
    print("No model identity columns in data — skipping model analysis.")

In [ ]:
# --- 3c. Conversation Length Analysis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution of n_turns
turn_counts = train_df['n_turns'].value_counts().sort_index()
axes[0].bar(turn_counts.index, turn_counts.values, color='#7E57C2')
axes[0].set_title('Distribution of Conversation Turns')
axes[0].set_xlabel('Number of Turns')
axes[0].set_ylabel('Count')
axes[0].set_xlim(0, min(15, turn_counts.index.max() + 1))

# Win rate by number of turns
turn_outcomes = train_df.groupby('n_turns')[TARGET_COLS].mean()
turn_outcomes_top = turn_outcomes[turn_outcomes.index <= 10]
turn_outcomes_top.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Win Rate by Number of Turns')
axes[1].set_xlabel('Number of Turns')
axes[1].set_ylabel('Win Rate')
axes[1].legend(CLASS_NAMES)
axes[1].axhline(y=1/3, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

pct_single = (train_df['n_turns'] == 1).mean() * 100
print(f"Single-turn conversations: {pct_single:.1f}%")

In [ ]:
# --- 3d. Text Length Distributions ---
train_df['prompt_len'] = train_df['prompt_text'].str.len()
train_df['response_a_len'] = train_df['response_a_text'].str.len()
train_df['response_b_len'] = train_df['response_b_text'].str.len()
train_df['len_diff'] = train_df['response_a_len'] - train_df['response_b_len']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Prompt length (log scale)
axes[0, 0].hist(train_df['prompt_len'].clip(upper=5000), bins=50, color='#78909C', alpha=0.8)
axes[0, 0].set_title('Prompt Length Distribution')
axes[0, 0].set_xlabel('Characters (clipped at 5000)')

# Response lengths comparison
axes[0, 1].hist(train_df['response_a_len'].clip(upper=5000), bins=50, alpha=0.6, label='Response A', color='#2196F3')
axes[0, 1].hist(train_df['response_b_len'].clip(upper=5000), bins=50, alpha=0.6, label='Response B', color='#FF5722')
axes[0, 1].set_title('Response Length Distributions')
axes[0, 1].set_xlabel('Characters (clipped at 5000)')
axes[0, 1].legend()

# Length difference by outcome
for i, (name, color) in enumerate(zip(CLASS_NAMES, colors)):
    mask = y_labels == i
    axes[1, 0].hist(train_df.loc[mask, 'len_diff'].clip(-3000, 3000), bins=50, 
                     alpha=0.5, label=name, color=color)
axes[1, 0].set_title('Response Length Difference (A - B) by Outcome')
axes[1, 0].set_xlabel('Length Difference (chars)')
axes[1, 0].legend()
axes[1, 0].axvline(x=0, color='black', linestyle='--', alpha=0.3)

# Box plot of length diff by outcome
data_for_box = [train_df.loc[y_labels == i, 'len_diff'].clip(-3000, 3000) for i in range(3)]
bp = axes[1, 1].boxplot(data_for_box, labels=CLASS_NAMES, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1, 1].set_title('Length Difference by Outcome')
axes[1, 1].set_ylabel('Length Diff (A - B)')
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['report_dir'] / 'text_lengths.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Median prompt length: {train_df['prompt_len'].median():.0f} chars")
print(f"Median response A length: {train_df['response_a_len'].median():.0f} chars")
print(f"Median response B length: {train_df['response_b_len'].median():.0f} chars")
print(f"\nMean length diff when A wins: {train_df.loc[y_labels==0, 'len_diff'].mean():.0f}")
print(f"Mean length diff when B wins: {train_df.loc[y_labels==1, 'len_diff'].mean():.0f}")
print(f"Mean length diff when tie:    {train_df.loc[y_labels==2, 'len_diff'].mean():.0f}")

In [ ]:
# --- 3e. Position Bias & Structural Analysis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Position bias check
a_win_rate = train_df['winner_model_a'].mean()
b_win_rate = train_df['winner_model_b'].mean()
tie_rate = train_df['winner_tie'].mean()
axes[0].bar(['A wins', 'B wins', 'Tie'], [a_win_rate, b_win_rate, tie_rate], color=colors)
axes[0].set_title('Position Bias Check: Overall Win Rates')
axes[0].set_ylabel('Rate')
axes[0].axhline(y=1/3, color='gray', linestyle='--', alpha=0.5, label='Uniform (1/3)')
axes[0].legend()
for i, v in enumerate([a_win_rate, b_win_rate, tie_rate]):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

# Structural features vs outcome
def has_code_block(text):
    return bool(re.search(r'```', str(text)))

def has_bullet_list(text):
    return bool(re.search(r'^\s*[-*]\s', str(text), re.MULTILINE))

train_df['a_has_code'] = train_df['response_a_text'].apply(has_code_block).astype(int)
train_df['b_has_code'] = train_df['response_b_text'].apply(has_code_block).astype(int)
train_df['a_has_bullets'] = train_df['response_a_text'].apply(has_bullet_list).astype(int)
train_df['b_has_bullets'] = train_df['response_b_text'].apply(has_bullet_list).astype(int)

# Code block analysis
code_scenarios = {
    'Both have code': (train_df['a_has_code'] == 1) & (train_df['b_has_code'] == 1),
    'Only A has code': (train_df['a_has_code'] == 1) & (train_df['b_has_code'] == 0),
    'Only B has code': (train_df['a_has_code'] == 0) & (train_df['b_has_code'] == 1),
    'Neither has code': (train_df['a_has_code'] == 0) & (train_df['b_has_code'] == 0),
}
scenario_rates = {}
for name, mask in code_scenarios.items():
    if mask.sum() > 0:
        scenario_rates[name] = train_df.loc[mask, TARGET_COLS].mean().values

x_pos = range(len(scenario_rates))
width = 0.25
for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    vals = [scenario_rates[s][i] for s in scenario_rates]
    axes[1].bar([p + i * width for p in x_pos], vals, width, label=cls_name, color=color)
axes[1].set_xticks([p + width for p in x_pos])
axes[1].set_xticklabels(scenario_rates.keys(), rotation=15, ha='right')
axes[1].set_title('Win Rate by Code Block Presence')
axes[1].set_ylabel('Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Position A win rate: {a_win_rate:.4f} ({a_win_rate-1/3:+.4f} vs uniform)")
print(f"Position B win rate: {b_win_rate:.4f} ({b_win_rate-1/3:+.4f} vs uniform)")
print(f"Responses with code blocks: A={train_df['a_has_code'].mean():.1%}, B={train_df['b_has_code'].mean():.1%}")

### EDA Key Findings

1. **Balanced classes**: ~35/34/31 split (A/B/Tie) — no severe imbalance
2. **Slight position bias**: A wins slightly more often than B (expected from Chatbot Arena)
3. **Verbosity signal**: Winners tend to have longer responses — length difference is informative but shouldn't be sole predictor
4. **75%+ single-turn**: Most conversations are single-turn; multi-turn is a minority case
5. **Code blocks**: When only one response has code, it correlates with winning
6. **No model identity at test time**: All features must be text-derived

## 4. Feature Engineering

In [ ]:
# --- 4a. Feature Engineering Helper Functions ---

def count_syllables(word: str) -> int:
    """Estimate syllable count using vowel-group heuristic."""
    word = word.lower().strip()
    if not word:
        return 0
    vowels = 'aeiouy'
    count = 0
    prev_vowel = False
    for char in word:
        is_vowel = char in vowels
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel
    # Adjust for silent e
    if word.endswith('e') and count > 1:
        count -= 1
    return max(count, 1)

def flesch_kincaid_grade(text: str) -> float:
    """Compute Flesch-Kincaid Grade Level."""
    sentences = sent_tokenize(str(text))
    words = str(text).split()
    n_sentences = max(len(sentences), 1)
    n_words = max(len(words), 1)
    n_syllables = sum(count_syllables(w) for w in words)
    return 0.39 * (n_words / n_sentences) + 11.8 * (n_syllables / n_words) - 15.59

def coleman_liau_index(text: str) -> float:
    """Compute Coleman-Liau Index."""
    words = str(text).split()
    sentences = sent_tokenize(str(text))
    n_words = max(len(words), 1)
    n_sentences = max(len(sentences), 1)
    n_chars = sum(len(w) for w in words)
    L = (n_chars / n_words) * 100  # avg letters per 100 words
    S = (n_sentences / n_words) * 100  # avg sentences per 100 words
    return 0.0588 * L - 0.296 * S - 15.8

def automated_readability_index(text: str) -> float:
    """Compute Automated Readability Index (ARI)."""
    words = str(text).split()
    sentences = sent_tokenize(str(text))
    n_words = max(len(words), 1)
    n_sentences = max(len(sentences), 1)
    n_chars = sum(len(w) for w in words)
    return 4.71 * (n_chars / n_words) + 0.5 * (n_words / n_sentences) - 21.43

def count_pattern(text: str, pattern: str) -> int:
    """Count regex pattern matches in text."""
    return len(re.findall(pattern, str(text), re.MULTILINE))

def count_phrase_matches(text: str, phrases: list) -> int:
    """Count occurrences of phrases in text (case-insensitive)."""
    text_lower = str(text).lower()
    return sum(text_lower.count(p.lower()) for p in phrases)

def repetition_ratio(text: str, n: int = 3) -> float:
    """Ratio of repeated n-grams to total n-grams."""
    words = str(text).lower().split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    counts = Counter(ngrams)
    repeated = sum(c - 1 for c in counts.values() if c > 1)
    return repeated / len(ngrams)

print("Feature engineering helpers defined.")

In [ ]:
# --- 4b. FeatureEngineer Class ---

class FeatureEngineer:
    """Extracts all hand-crafted features from prompt/response text."""
    
    def __init__(self, config: dict):
        self.config = config
        self.feature_names = []
    
    def _text_stats(self, text: str) -> dict:
        """Compute text statistics for a single text."""
        text = str(text)
        words = text.split()
        sentences = sent_tokenize(text) if len(text) > 0 else ['']
        n_words = len(words)
        n_sentences = max(len(sentences), 1)
        
        unique_words = set(w.lower() for w in words)
        stop_count = sum(1 for w in words if w.lower() in STOPWORDS)
        
        return {
            'char_len': len(text),
            'word_count': n_words,
            'sentence_count': n_sentences,
            'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
            'avg_sentence_length': n_words / n_sentences,
            'vocab_richness': len(unique_words) / max(n_words, 1),
            'stopword_ratio': stop_count / max(n_words, 1),
        }
    
    def _structural_features(self, text: str) -> dict:
        """Compute structural/formatting features."""
        text = str(text)
        paragraphs = [p for p in text.split('\n\n') if p.strip()]
        return {
            'paragraph_count': len(paragraphs),
            'has_code_block': int(bool(re.search(r'```', text))),
            'code_block_count': len(re.findall(r'```', text)) // 2,
            'has_bullet_list': int(bool(re.search(r'^\s*[-*]\s', text, re.MULTILINE))),
            'bullet_count': count_pattern(text, r'^\s*[-*]\s'),
            'has_numbered_list': int(bool(re.search(r'^\s*\d+\.\s', text, re.MULTILINE))),
            'numbered_list_count': count_pattern(text, r'^\s*\d+\.\s'),
            'has_markdown_headers': int(bool(re.search(r'^#+\s', text, re.MULTILINE))),
            'has_bold': int(bool(re.search(r'\*\*[^*]+\*\*', text))),
            'has_links': int(bool(re.search(r'https?://', text))),
            'newline_count': text.count('\n'),
        }
    
    def _quality_signals(self, text: str, prompt_text: str) -> dict:
        """Compute response quality signal features."""
        text = str(text)
        prompt = str(prompt_text)
        hedging = self.config.get('hedging_words', [])
        confidence = self.config.get('confidence_words', [])
        examples = self.config.get('example_indicators', [])
        
        n_words = max(len(text.split()), 1)
        special_chars = sum(1 for c in text if not c.isalnum() and not c.isspace())
        
        return {
            'hedging_count': count_phrase_matches(text, hedging),
            'confidence_count': count_phrase_matches(text, confidence),
            'example_count': count_phrase_matches(text, examples),
            'repetition_ratio': repetition_ratio(text),
            'special_char_ratio': special_chars / max(len(text), 1),
            'starts_with_i': int(text.strip().startswith('I ')),
        }
    
    def extract_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract all hand-crafted features from a DataFrame."""
        features = pd.DataFrame(index=df.index)
        cfg = self.config
        
        logger.info("Extracting text statistics...")
        # --- Text Statistics ---
        for prefix, col in [('prompt', 'prompt_text'), ('resp_a', 'response_a_text'), ('resp_b', 'response_b_text')]:
            stats = df[col].apply(self._text_stats).apply(pd.Series)
            stats.columns = [f'{prefix}_{c}' for c in stats.columns]
            features = pd.concat([features, stats], axis=1)
        
        # Difference features (A - B)
        for metric in ['char_len', 'word_count', 'sentence_count', 'avg_word_length', 
                        'avg_sentence_length', 'vocab_richness', 'stopword_ratio']:
            features[f'diff_{metric}'] = features[f'resp_a_{metric}'] - features[f'resp_b_{metric}']
        
        # Ratio features
        features['len_ratio'] = features['resp_a_char_len'] / (features['resp_b_char_len'] + 1)
        features['word_count_ratio'] = features['resp_a_word_count'] / (features['resp_b_word_count'] + 1)
        features['log_len_ratio'] = np.log1p(features['len_ratio'])
        
        # --- Readability ---
        if cfg.get('readability_enabled', True):
            logger.info("Extracting readability metrics...")
            for prefix, col in [('resp_a', 'response_a_text'), ('resp_b', 'response_b_text')]:
                features[f'{prefix}_fk_grade'] = df[col].apply(flesch_kincaid_grade)
                features[f'{prefix}_cli'] = df[col].apply(coleman_liau_index)
                features[f'{prefix}_ari'] = df[col].apply(automated_readability_index)
            for metric in ['fk_grade', 'cli', 'ari']:
                features[f'diff_{metric}'] = features[f'resp_a_{metric}'] - features[f'resp_b_{metric}']
        
        # --- Structural Features ---
        if cfg.get('structural_features_enabled', True):
            logger.info("Extracting structural features...")
            for prefix, col in [('resp_a', 'response_a_text'), ('resp_b', 'response_b_text')]:
                struct = df[col].apply(self._structural_features).apply(pd.Series)
                struct.columns = [f'{prefix}_{c}' for c in struct.columns]
                features = pd.concat([features, struct], axis=1)
            
            # Formatting richness composite
            fmt_cols_a = ['resp_a_has_code_block', 'resp_a_has_bullet_list', 'resp_a_has_numbered_list',
                          'resp_a_has_markdown_headers', 'resp_a_has_bold', 'resp_a_has_links']
            fmt_cols_b = [c.replace('resp_a_', 'resp_b_') for c in fmt_cols_a]
            features['resp_a_formatting_richness'] = features[fmt_cols_a].sum(axis=1)
            features['resp_b_formatting_richness'] = features[fmt_cols_b].sum(axis=1)
            features['diff_formatting_richness'] = features['resp_a_formatting_richness'] - features['resp_b_formatting_richness']
            
            # Structural diffs
            for metric in ['paragraph_count', 'code_block_count', 'bullet_count', 'newline_count']:
                features[f'diff_{metric}'] = features[f'resp_a_{metric}'] - features[f'resp_b_{metric}']
        
        # --- Quality Signals ---
        if cfg.get('quality_signals_enabled', True):
            logger.info("Extracting quality signals...")
            for prefix, col in [('resp_a', 'response_a_text'), ('resp_b', 'response_b_text')]:
                quality = df.apply(lambda row: self._quality_signals(row[col], row['prompt_text']), axis=1).apply(pd.Series)
                quality.columns = [f'{prefix}_{c}' for c in quality.columns]
                features = pd.concat([features, quality], axis=1)
            
            for metric in ['hedging_count', 'confidence_count', 'example_count', 'repetition_ratio']:
                features[f'diff_{metric}'] = features[f'resp_a_{metric}'] - features[f'resp_b_{metric}']
            
            # Prompt features
            features['prompt_is_question'] = df['last_prompt'].str.strip().str.endswith('?').astype(int)
            features['prompt_complexity'] = (features['prompt_word_count'] + 
                                             features['prompt_sentence_count'] + 
                                             features['prompt_vocab_richness'])
            features['prompt_has_code'] = df['prompt_text'].apply(lambda x: int(bool(re.search(r'```', str(x)))))
        
        # --- Conversation Features ---
        logger.info("Extracting conversation features...")
        features['n_turns'] = df['n_turns']
        features['is_multi_turn'] = (df['n_turns'] > 1).astype(int)
        features['last_prompt_len'] = df['last_prompt'].str.len()
        features['last_resp_a_len'] = df['last_response_a'].str.len()
        features['last_resp_b_len'] = df['last_response_b'].str.len()
        features['diff_last_resp_len'] = features['last_resp_a_len'] - features['last_resp_b_len']
        
        # Turn-level consistency for multi-turn
        def turn_length_std(parsed_list):
            lengths = [len(str(t)) for t in parsed_list]
            return np.std(lengths) if len(lengths) > 1 else 0
        
        features['resp_a_length_consistency'] = df['response_a_parsed'].apply(turn_length_std)
        features['resp_b_length_consistency'] = df['response_b_parsed'].apply(turn_length_std)
        
        # --- Bias Mitigation Features ---
        if cfg.get('bias_mitigation_enabled', True):
            logger.info("Extracting bias mitigation features...")
            features['abs_len_diff'] = features['diff_char_len'].abs()
            features['max_response_len'] = features[['resp_a_char_len', 'resp_b_char_len']].max(axis=1)
            features['min_response_len'] = features[['resp_a_char_len', 'resp_b_char_len']].min(axis=1)
            features['len_ratio_symmetric'] = features['min_response_len'] / (features['max_response_len'] + 1)
            features['longer_is_a'] = (features['resp_a_char_len'] > features['resp_b_char_len']).astype(int)
            features['abs_formatting_diff'] = features['diff_formatting_richness'].abs()
            features['both_have_code'] = (features['resp_a_has_code_block'] & features['resp_b_has_code_block']).astype(int)
            features['neither_has_code'] = ((1 - features['resp_a_has_code_block']) & (1 - features['resp_b_has_code_block'])).astype(int)
        
        # --- Clean up ---
        features = features.replace([np.inf, -np.inf], np.nan)
        features = features.fillna(0)
        
        self.feature_names = list(features.columns)
        logger.info(f"Extracted {len(self.feature_names)} hand-crafted features.")
        return features

print("FeatureEngineer class defined.")

In [ ]:
# --- 4c. Extract Hand-crafted Features ---
fe = FeatureEngineer(config=features_cfg)

print("Extracting train features...")
X_train_handcrafted = fe.extract_features(train_df)

print("Extracting test features...")
X_test_handcrafted = fe.extract_features(test_df)

print(f"\nHand-crafted features: {X_train_handcrafted.shape[1]}")
print(f"Train shape: {X_train_handcrafted.shape}")
print(f"Test shape:  {X_test_handcrafted.shape}")
print(f"\nFeature sample (first 5 rows, first 10 cols):")
X_train_handcrafted.iloc[:5, :10]

### 4b. Embedding Features

In [ ]:
# --- 4d. Compute Sentence Embeddings ---
EMBEDDINGS_AVAILABLE = False

if HAS_SENTENCE_TRANSFORMERS:
    embedding_model_name = features_cfg['embedding_model']
    max_text_len = features_cfg['max_text_length_for_embedding']
    
    try:
        logger.info(f"Loading embedding model: {embedding_model_name}")
        embed_model = SentenceTransformer(embedding_model_name)
        EMBEDDINGS_AVAILABLE = True
    except Exception as e:
        logger.warning(f"Could not load embedding model (no internet?): {e}")
        print(f"⚠ Embedding model unavailable — skipping embedding features.")

if EMBEDDINGS_AVAILABLE:
    def compute_embeddings(texts: list, model, max_length: int = 512, batch_size: int = 64) -> np.ndarray:
        """Compute sentence embeddings with truncation."""
        truncated = [str(t)[:max_length] for t in texts]
        embeddings = model.encode(truncated, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
        return embeddings

    # Compute embeddings for train
    logger.info("Computing train embeddings...")
    train_prompt_emb = compute_embeddings(train_df['prompt_text'].tolist(), embed_model, max_text_len)
    train_resp_a_emb = compute_embeddings(train_df['response_a_text'].tolist(), embed_model, max_text_len)
    train_resp_b_emb = compute_embeddings(train_df['response_b_text'].tolist(), embed_model, max_text_len)

    # Compute embeddings for test
    logger.info("Computing test embeddings...")
    test_prompt_emb = compute_embeddings(test_df['prompt_text'].tolist(), embed_model, max_text_len)
    test_resp_a_emb = compute_embeddings(test_df['response_a_text'].tolist(), embed_model, max_text_len)
    test_resp_b_emb = compute_embeddings(test_df['response_b_text'].tolist(), embed_model, max_text_len)

    print(f"Embedding shape: {train_prompt_emb.shape}")
    print("All embeddings computed.")
else:
    print("Skipping embeddings — model not available (no internet access).")
    print("Pipeline will use hand-crafted features only.")

In [ ]:
# --- 4e. Similarity Features + PCA Reduction ---
if EMBEDDINGS_AVAILABLE:
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim

    def compute_similarity_features(prompt_emb, resp_a_emb, resp_b_emb):
        """Compute pairwise cosine similarity features from embeddings."""
        n = len(prompt_emb)
        features = pd.DataFrame()
        
        # Cosine similarities (row-wise)
        features['sim_prompt_a'] = np.array([cos_sim(prompt_emb[i:i+1], resp_a_emb[i:i+1])[0, 0] for i in range(n)])
        features['sim_prompt_b'] = np.array([cos_sim(prompt_emb[i:i+1], resp_b_emb[i:i+1])[0, 0] for i in range(n)])
        features['sim_a_b'] = np.array([cos_sim(resp_a_emb[i:i+1], resp_b_emb[i:i+1])[0, 0] for i in range(n)])
        
        # Derived
        features['prompt_relevance_diff'] = features['sim_prompt_a'] - features['sim_prompt_b']
        features['prompt_relevance_ratio'] = features['sim_prompt_a'] / (features['sim_prompt_b'] + 1e-8)
        
        # Euclidean distance between responses
        features['euclidean_dist_a_b'] = np.linalg.norm(resp_a_emb - resp_b_emb, axis=1)
        
        return features

    logger.info("Computing similarity features...")
    train_sim_features = compute_similarity_features(train_prompt_emb, train_resp_a_emb, train_resp_b_emb)
    test_sim_features = compute_similarity_features(test_prompt_emb, test_resp_a_emb, test_resp_b_emb)

    # PCA on response difference embeddings
    logger.info("Applying PCA on response difference embeddings...")
    n_pca_resp = features_cfg['pca_components_response_diff']
    n_pca_prompt = features_cfg['pca_components_prompt']

    train_resp_diff_emb = train_resp_a_emb - train_resp_b_emb
    test_resp_diff_emb = test_resp_a_emb - test_resp_b_emb

    pca_resp = PCA(n_components=n_pca_resp, random_state=SEED)
    train_pca_resp = pca_resp.fit_transform(train_resp_diff_emb)
    test_pca_resp = pca_resp.transform(test_resp_diff_emb)

    pca_prompt = PCA(n_components=n_pca_prompt, random_state=SEED)
    train_pca_prompt = pca_prompt.fit_transform(train_prompt_emb)
    test_pca_prompt = pca_prompt.transform(test_prompt_emb)

    # Convert to DataFrames
    train_pca_resp_df = pd.DataFrame(train_pca_resp, columns=[f'pca_resp_diff_{i}' for i in range(n_pca_resp)], index=train_df.index)
    test_pca_resp_df = pd.DataFrame(test_pca_resp, columns=[f'pca_resp_diff_{i}' for i in range(n_pca_resp)], index=test_df.index)
    train_pca_prompt_df = pd.DataFrame(train_pca_prompt, columns=[f'pca_prompt_{i}' for i in range(n_pca_prompt)], index=train_df.index)
    test_pca_prompt_df = pd.DataFrame(test_pca_prompt, columns=[f'pca_prompt_{i}' for i in range(n_pca_prompt)], index=test_df.index)

    print(f"Similarity features: {train_sim_features.shape[1]}")
    print(f"PCA response diff features: {n_pca_resp}")
    print(f"PCA prompt features: {n_pca_prompt}")
    print(f"PCA explained variance (resp diff): {pca_resp.explained_variance_ratio_.sum():.3f}")
    print(f"PCA explained variance (prompt): {pca_prompt.explained_variance_ratio_.sum():.3f}")
else:
    print("Skipping similarity & PCA features — embeddings not available.")

In [ ]:
# --- 4f. Combine All Features ---
train_parts = [X_train_handcrafted.reset_index(drop=True)]
test_parts = [X_test_handcrafted.reset_index(drop=True)]

if EMBEDDINGS_AVAILABLE:
    train_parts.extend([
        train_sim_features.reset_index(drop=True),
        train_pca_resp_df.reset_index(drop=True),
        train_pca_prompt_df.reset_index(drop=True),
    ])
    test_parts.extend([
        test_sim_features.reset_index(drop=True),
        test_pca_resp_df.reset_index(drop=True),
        test_pca_prompt_df.reset_index(drop=True),
    ])

X_train = pd.concat(train_parts, axis=1)
X_test = pd.concat(test_parts, axis=1)

# Final cleanup
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

ALL_FEATURE_NAMES = list(X_train.columns)

logger.info(f"Total features: {len(ALL_FEATURE_NAMES)}")
print(f"Final feature matrix:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  Total features: {len(ALL_FEATURE_NAMES)}")
print(f"\nFeature categories:")
print(f"  Hand-crafted: {X_train_handcrafted.shape[1]}")
if EMBEDDINGS_AVAILABLE:
    print(f"  Similarity:   {train_sim_features.shape[1]}")
    print(f"  PCA (resp):   {n_pca_resp}")
    print(f"  PCA (prompt): {n_pca_prompt}")
else:
    print(f"  Embeddings:   SKIPPED (no internet)")

## 5. Model Development

In [ ]:
# --- 5a. Cross-Validation Setup ---
N_FOLDS = models_cfg['cv_folds']
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def evaluate_cv(model_fn, X, y_labels, y_onehot, skf, model_name="model"):
    """Run stratified K-fold CV and return OOF predictions + scores."""
    oof_preds = np.zeros((len(X), 3))
    fold_scores = []
    fold_models = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_labels)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_labels[train_idx], y_labels[val_idx]
        y_val_onehot = y_onehot[val_idx]
        
        model = model_fn(fold)
        
        if hasattr(model, 'fit_with_eval'):
            model.fit_with_eval(X_tr, y_tr, X_val, y_val)
        else:
            model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_val)
        oof_preds[val_idx] = preds
        
        score = log_loss(y_val_onehot, preds)
        fold_scores.append(score)
        fold_models.append(model)
        logger.info(f"  {model_name} Fold {fold+1}/{N_FOLDS}: log_loss = {score:.5f}")
    
    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    logger.info(f"  {model_name} CV: {mean_score:.5f} +/- {std_score:.5f}")
    
    return {
        'oof_preds': oof_preds,
        'fold_scores': fold_scores,
        'fold_models': fold_models,
        'mean_score': mean_score,
        'std_score': std_score,
    }

print(f"CV setup: {N_FOLDS}-fold stratified")
print(f"Uniform baseline log loss: {log_loss(y_train, np.full_like(y_train, 1/3, dtype=float)):.5f}")

In [ ]:
# --- 5b. Baseline: Logistic Regression ---
logger.info("Training Logistic Regression baseline...")
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)

def make_logistic(fold):
    return LogisticRegression(
        multi_class='multinomial', solver='lbfgs',
        max_iter=models_cfg['logistic']['max_iter'],
        C=1.0, random_state=SEED
    )

lr_results = evaluate_cv(make_logistic, X_train_scaled, y_labels, y_train, skf, "LogisticRegression")
print(f"\nLogistic Regression CV: {lr_results['mean_score']:.5f} +/- {lr_results['std_score']:.5f}")

In [ ]:
# --- 5c. LightGBM with Default Params ---

class LGBMWrapper:
    """LightGBM wrapper with early stopping support."""
    def __init__(self, params, seed=42):
        self.params = {**params, 'random_state': seed}
        self.model = None
    
    def fit_with_eval(self, X_tr, y_tr, X_val, y_val):
        self.model = lgb.LGBMClassifier(**self.params)
        self.model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
        )
    
    def fit(self, X, y):
        self.model = lgb.LGBMClassifier(**self.params)
        self.model.fit(X, y)
    
    def predict_proba(self, X):
        return self.model.predict_proba(X)

lgbm_default = {**models_cfg['lgbm']['fixed_params'], **models_cfg['lgbm']['default_params']}

def make_lgbm_default(fold):
    wrapper = LGBMWrapper(lgbm_default, seed=SEED + fold)
    return wrapper

logger.info("Training LightGBM with default params...")
lgbm_default_results = evaluate_cv(make_lgbm_default, X_train, y_labels, y_train, skf, "LightGBM-default")
print(f"\nLightGBM (default) CV: {lgbm_default_results['mean_score']:.5f} +/- {lgbm_default_results['std_score']:.5f}")

In [ ]:
# --- 5d. Optuna Hyperparameter Tuning — LightGBM ---
N_OPTUNA_TRIALS = models_cfg['optuna_trials']
lgbm_fixed = models_cfg['lgbm']['fixed_params']
lgbm_space = models_cfg['lgbm']['search_space']

def objective_lgbm(trial):
    params = {
        **lgbm_fixed,
        'n_estimators': trial.suggest_int('n_estimators', *lgbm_space['n_estimators']),
        'learning_rate': trial.suggest_float('learning_rate', *lgbm_space['learning_rate'], log=True),
        'max_depth': trial.suggest_int('max_depth', *lgbm_space['max_depth']),
        'num_leaves': trial.suggest_int('num_leaves', *lgbm_space['num_leaves']),
        'min_child_samples': trial.suggest_int('min_child_samples', *lgbm_space['min_child_samples']),
        'subsample': trial.suggest_float('subsample', *lgbm_space['subsample']),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *lgbm_space['colsample_bytree']),
        'reg_alpha': trial.suggest_float('reg_alpha', *lgbm_space['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *lgbm_space['reg_lambda'], log=True),
    }
    
    fold_scores = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_labels)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_labels[train_idx], y_labels[val_idx]
        
        model = lgb.LGBMClassifier(**params, random_state=SEED + fold)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
        )
        preds = model.predict_proba(X_val)
        fold_scores.append(log_loss(y_train[val_idx], preds))
        
        # Pruning
        trial.report(np.mean(fold_scores), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(fold_scores)

logger.info(f"Starting Optuna LightGBM tuning ({N_OPTUNA_TRIALS} trials)...")
study_lgbm = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study_lgbm.optimize(objective_lgbm, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

best_lgbm_params = {**lgbm_fixed, **study_lgbm.best_params}
print(f"\nBest LightGBM CV log loss: {study_lgbm.best_value:.5f}")
print(f"Best params: {study_lgbm.best_params}")

In [ ]:
# --- 5e. Optuna Hyperparameter Tuning — XGBoost ---
xgb_fixed = models_cfg['xgb']['fixed_params']
xgb_space = models_cfg['xgb']['search_space']

class XGBWrapper:
    """XGBoost wrapper with early stopping support."""
    def __init__(self, params, seed=42):
        self.params = {**params, 'random_state': seed}
        self.model = None
    
    def fit_with_eval(self, X_tr, y_tr, X_val, y_val):
        self.model = xgb.XGBClassifier(**self.params)
        self.model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    
    def predict_proba(self, X):
        return self.model.predict_proba(X)

def objective_xgb(trial):
    params = {
        **xgb_fixed,
        'n_estimators': trial.suggest_int('n_estimators', *xgb_space['n_estimators']),
        'learning_rate': trial.suggest_float('learning_rate', *xgb_space['learning_rate'], log=True),
        'max_depth': trial.suggest_int('max_depth', *xgb_space['max_depth']),
        'min_child_weight': trial.suggest_int('min_child_weight', *xgb_space['min_child_weight']),
        'subsample': trial.suggest_float('subsample', *xgb_space['subsample']),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *xgb_space['colsample_bytree']),
        'reg_alpha': trial.suggest_float('reg_alpha', *xgb_space['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *xgb_space['reg_lambda'], log=True),
        'gamma': trial.suggest_float('gamma', *xgb_space['gamma']),
        'early_stopping_rounds': 50,
    }
    
    fold_scores = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_labels)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_labels[train_idx], y_labels[val_idx]
        
        model = xgb.XGBClassifier(**params, random_state=SEED + fold)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        preds = model.predict_proba(X_val)
        fold_scores.append(log_loss(y_train[val_idx], preds))
        
        trial.report(np.mean(fold_scores), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(fold_scores)

logger.info(f"Starting Optuna XGBoost tuning ({N_OPTUNA_TRIALS} trials)...")
study_xgb = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study_xgb.optimize(objective_xgb, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

best_xgb_params = {**xgb_fixed, **{k: v for k, v in study_xgb.best_params.items()}}
print(f"\nBest XGBoost CV log loss: {study_xgb.best_value:.5f}")
print(f"Best params: {study_xgb.best_params}")

In [ ]:
# --- 5f. Train Final Models with Best Params ---

# Retrain LightGBM with best params
def make_lgbm_tuned(fold):
    return LGBMWrapper(best_lgbm_params, seed=SEED + fold)

logger.info("Training tuned LightGBM...")
lgbm_tuned_results = evaluate_cv(make_lgbm_tuned, X_train, y_labels, y_train, skf, "LightGBM-tuned")

# Retrain XGBoost with best params
def make_xgb_tuned(fold):
    params = {**best_xgb_params, 'early_stopping_rounds': 50}
    return XGBWrapper(params, seed=SEED + fold)

logger.info("Training tuned XGBoost...")
xgb_tuned_results = evaluate_cv(make_xgb_tuned, X_train, y_labels, y_train, skf, "XGBoost-tuned")

# Save models
model_dir = PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['model_dir']
joblib.dump(lgbm_tuned_results['fold_models'], model_dir / 'lgbm_fold_models.pkl')
joblib.dump(xgb_tuned_results['fold_models'], model_dir / 'xgb_fold_models.pkl')

# Model comparison table
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
uniform_ll = log_loss(y_train, np.full_like(y_train, 1/3, dtype=float))
results_table = {
    'Uniform Baseline': (uniform_ll, 0.0),
    'Logistic Regression': (lr_results['mean_score'], lr_results['std_score']),
    'LightGBM (default)': (lgbm_default_results['mean_score'], lgbm_default_results['std_score']),
    'LightGBM (tuned)': (lgbm_tuned_results['mean_score'], lgbm_tuned_results['std_score']),
    'XGBoost (tuned)': (xgb_tuned_results['mean_score'], xgb_tuned_results['std_score']),
}
print(f"{'Model':<25} {'CV Log Loss':>15} {'Std':>10}")
print("-"*50)
for name, (score, std) in results_table.items():
    print(f"{name:<25} {score:>15.5f} {std:>10.5f}")
print("="*60)

## 6. Ensemble

In [ ]:
# --- 6a. Ensemble Weight Optimization ---

oof_preds_list = [
    lgbm_tuned_results['oof_preds'],
    xgb_tuned_results['oof_preds'],
]
model_names_for_ensemble = ['LightGBM', 'XGBoost']

def ensemble_log_loss(weights, oof_list, y_true):
    """Compute log loss for weighted ensemble of OOF predictions."""
    weights = np.array(weights)
    weights = weights / weights.sum()
    blended = sum(w * p for w, p in zip(weights, oof_list))
    return log_loss(y_true, blended)

# Optimize weights
n_models = len(oof_preds_list)
result = minimize(
    ensemble_log_loss,
    x0=[1 / n_models] * n_models,
    args=(oof_preds_list, y_train),
    method='Nelder-Mead',
)
optimal_weights = result.x / result.x.sum()

# Compute ensemble OOF predictions
ensemble_oof = sum(w * p for w, p in zip(optimal_weights, oof_preds_list))
ensemble_cv_score = log_loss(y_train, ensemble_oof)

print("Ensemble Weights:")
for name, w in zip(model_names_for_ensemble, optimal_weights):
    print(f"  {name}: {w:.4f}")
print(f"\nEnsemble CV Log Loss: {ensemble_cv_score:.5f}")
print(f"Best single model:   {min(lgbm_tuned_results['mean_score'], xgb_tuned_results['mean_score']):.5f}")
print(f"Improvement:         {min(lgbm_tuned_results['mean_score'], xgb_tuned_results['mean_score']) - ensemble_cv_score:.5f}")

## 7. Evaluation and Analysis

In [ ]:
# --- 7a. Confusion Matrix & Classification Report ---
ensemble_preds_hard = np.argmax(ensemble_oof, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_labels, ensemble_preds_hard)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Ensemble Confusion Matrix (OOF)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Normalized confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Normalized Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['report_dir'] / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nClassification Report:")
print(classification_report(y_labels, ensemble_preds_hard, target_names=CLASS_NAMES))

# Per-class log loss
for i, name in enumerate(CLASS_NAMES):
    class_ll = log_loss(y_train[:, i], ensemble_oof[:, i], labels=[0, 1])
    print(f"  {name} log loss: {class_ll:.5f}")

In [ ]:
# --- 7b. SHAP Feature Importance ---
logger.info("Computing SHAP values (subsample for speed)...")

# Use the first fold's LightGBM model
lgbm_model_for_shap = lgbm_tuned_results['fold_models'][0].model
shap_sample_size = min(1000, len(X_train))
X_shap = X_train.iloc[:shap_sample_size]

explainer = shap.TreeExplainer(lgbm_model_for_shap)
shap_values = explainer.shap_values(X_shap)

# Summary plot (all classes)
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_shap, class_names=CLASS_NAMES, max_display=20, show=False)
plt.title('SHAP Feature Importance (Top 20)')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['report_dir'] / 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

# Mean absolute SHAP values for feature ranking
# Handle both old (list of arrays) and new (3D array) SHAP formats
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    # 3D array: (n_samples, n_features, n_classes)
    mean_shap = np.mean(np.abs(shap_values), axis=(0, 2))
shap_importance = pd.Series(mean_shap, index=X_train.columns).sort_values(ascending=False)

print("Top 20 features by SHAP importance:")
for i, (feat, imp) in enumerate(shap_importance.head(20).items()):
    print(f"  {i+1:2d}. {feat:<40s} {imp:.5f}")

In [ ]:
# --- 7c. Feature Importance Comparison (LightGBM vs SHAP) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# LightGBM built-in importance (gain)
lgbm_importance = pd.Series(
    lgbm_model_for_shap.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

lgbm_importance.head(20).plot(kind='barh', ax=axes[0], color='#5C6BC0')
axes[0].set_title('LightGBM Feature Importance (Gain)')
axes[0].invert_yaxis()

# SHAP importance
shap_importance.head(20).plot(kind='barh', ax=axes[1], color='#26A69A')
axes[1].set_title('SHAP Feature Importance (Mean |SHAP|)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['report_dir'] / 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

# Feature category analysis
def categorize_feature(name):
    if name.startswith('pca_'): return 'PCA Embedding'
    if name.startswith('sim_') or name.startswith('euclidean') or name.startswith('prompt_relevance'): return 'Similarity'
    if any(x in name for x in ['fk_grade', 'cli', 'ari']): return 'Readability'
    if any(x in name for x in ['code_block', 'bullet', 'markdown', 'bold', 'link', 'paragraph', 'newline', 'formatting', 'numbered']): return 'Structural'
    if any(x in name for x in ['hedging', 'confidence', 'example', 'repetition', 'special_char', 'starts_with']): return 'Quality'
    if any(x in name for x in ['n_turns', 'multi_turn', 'consistency', 'last_']): return 'Conversation'
    if any(x in name for x in ['abs_', 'symmetric', 'longer_is', 'both_have', 'neither']): return 'Bias Mitigation'
    return 'Text Statistics'

category_importance = shap_importance.groupby(shap_importance.index.map(categorize_feature)).sum()
category_importance = category_importance.sort_values(ascending=False)

print("\nFeature importance by category:")
for cat, imp in category_importance.items():
    pct = imp / category_importance.sum() * 100
    print(f"  {cat:<20s}: {imp:.4f} ({pct:.1f}%)")

## 8. Prediction & Submission

In [ ]:
# --- 8a. Generate Test Predictions ---

def predict_ensemble(models_dict, weights, X_test):
    """Generate weighted ensemble predictions on test data."""
    all_preds = []
    for model_name, fold_models in models_dict.items():
        fold_preds = []
        for m in fold_models:
            if hasattr(m, 'model'):
                fold_preds.append(m.model.predict_proba(X_test))
            else:
                fold_preds.append(m.predict_proba(X_test))
        # Average across folds
        avg_pred = np.mean(fold_preds, axis=0)
        all_preds.append(avg_pred)
    
    # Weighted ensemble
    final_preds = sum(w * p for w, p in zip(weights, all_preds))
    return final_preds

models_dict = {
    'LightGBM': lgbm_tuned_results['fold_models'],
    'XGBoost': xgb_tuned_results['fold_models'],
}

logger.info("Generating test predictions...")
test_preds = predict_ensemble(models_dict, optimal_weights, X_test)

# Safety: clip probabilities and normalize
test_preds = np.clip(test_preds, 1e-7, 1 - 1e-7)
test_preds = test_preds / test_preds.sum(axis=1, keepdims=True)

print("Test predictions:")
print(pd.DataFrame(test_preds, columns=CLASS_NAMES))
print(f"\nRow sums (should be 1.0): {test_preds.sum(axis=1)}")

In [ ]:
# --- 8b. Create Submission & Save Artifacts ---
prediction_dir = PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['prediction_dir']

submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': test_preds[:, 0],
    'winner_model_b': test_preds[:, 1],
    'winner_tie': test_preds[:, 2],
})

# Verify format matches sample
assert list(submission.columns) == list(submission_df.columns), "Column mismatch!"
assert len(submission) == len(submission_df), "Row count mismatch!"

# Save submission to root submission.csv (Kaggle expects this)
submission.to_csv('submission.csv', index=False)
# Also save a copy to predictions dir
submission.to_csv(prediction_dir / 'submission.csv', index=False)
print("Submission saved to submission.csv")
print(submission)

# Save artifacts
model_dir = PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['model_dir']
joblib.dump(fe, model_dir / 'feature_engineer.pkl')
joblib.dump(scaler, model_dir / 'scaler.pkl')
joblib.dump({'pca_resp': pca_resp, 'pca_prompt': pca_prompt}, model_dir / 'pca_models.pkl')
joblib.dump(optimal_weights, model_dir / 'ensemble_weights.pkl')

# Save experiment config
import shutil
shutil.copytree(PROJECT_ROOT / 'configs', PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['log_dir'] / 'configs_used', dirs_exist_ok=True)

# Save CV results
cv_results = {
    'logistic_regression': lr_results['mean_score'],
    'lgbm_default': lgbm_default_results['mean_score'],
    'lgbm_tuned': lgbm_tuned_results['mean_score'],
    'xgb_tuned': xgb_tuned_results['mean_score'],
    'ensemble': ensemble_cv_score,
    'ensemble_weights': optimal_weights.tolist(),
    'best_lgbm_params': {k: (v if not isinstance(v, np.generic) else v.item()) for k, v in best_lgbm_params.items()},
    'best_xgb_params': {k: (v if not isinstance(v, np.generic) else v.item()) for k, v in best_xgb_params.items()},
}
with open(PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['log_dir'] / 'cv_results.json', 'w') as f:
    json.dump(cv_results, f, indent=2)

logger.info("All artifacts saved.")
print("\nArtifacts saved to outputs/models/ and outputs/logs/")

## 9. Results Summary

In [ ]:
# --- 9a. Performance Summary Dashboard ---
uniform_ll = log_loss(y_train, np.full_like(y_train, 1/3, dtype=float))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Model comparison bar chart
model_scores = {
    'Uniform': uniform_ll,
    'Logistic': lr_results['mean_score'],
    'LGBM\n(default)': lgbm_default_results['mean_score'],
    'LGBM\n(tuned)': lgbm_tuned_results['mean_score'],
    'XGB\n(tuned)': xgb_tuned_results['mean_score'],
    'Ensemble': ensemble_cv_score,
}
bar_colors = ['#BDBDBD', '#90CAF9', '#66BB6A', '#43A047', '#FF7043', '#AB47BC']
bars = axes[0].bar(model_scores.keys(), model_scores.values(), color=bar_colors)
axes[0].set_title('Model Comparison: CV Log Loss')
axes[0].set_ylabel('Log Loss (lower is better)')
for bar, score in zip(bars, model_scores.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, 
                 f'{score:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Feature category pie chart
category_importance.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Feature Importance by Category')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / paths_cfg['output_dir'] / paths_cfg['report_dir'] / 'results_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()

# Improvement summary
best_single = min(lgbm_tuned_results['mean_score'], xgb_tuned_results['mean_score'])
print("="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"Uniform baseline:     {uniform_ll:.5f}")
print(f"Best single model:    {best_single:.5f} ({(1 - best_single/uniform_ll)*100:.1f}% improvement)")
print(f"Ensemble:             {ensemble_cv_score:.5f} ({(1 - ensemble_cv_score/uniform_ll)*100:.1f}% improvement)")
print(f"Total features used:  {len(ALL_FEATURE_NAMES)}")
print(f"Training samples:     {len(X_train)}")
print(f"Test samples:         {len(X_test)}")
print(f"Experiment:           {experiment_cfg['name']}")
print(f"Random seed:          {SEED}")
print("="*60)